# SATP Methodology: Four-Step Circumplex Validation

This notebook runs the complete four-step circumplex validation procedure on the SATP v1.5 dataset:

1. **Step 1**: Circular order test (Tracey's RTHOR)
2. **Step 2**: Structural Equation Modelling (Browne's CircE)
3. **Step 3**: Structural Summary Method (SSM) analysis
4. **Step 4**: Congruence testing (cosine similarity and Procrustes)

In [1]:
import warnings
import sys
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rthor

import soundscapy as sspy
from soundscapy.satp import fit_circe, CircEResults
import soundscapy.r_wrapper as sspyr

from circumplex import ssm_analyze

warnings.filterwarnings("ignore", category=UserWarning, module="soundscapy")

# Eagerly initialise the R session before any parallel work (fixes rpy2 conversion rules error)
sspyr.get_r_session()

# Set up project paths — notebook lives in computation/, project root is one level up
NOTEBOOK_DIR = Path(__file__).parent if "__file__" in dir() else Path.cwd()
PROJECT_DIR = (
    NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "computation" else NOTEBOOK_DIR
)

# Add project root to sys.path so we can import computation.scm_inst
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from computation.scm_inst import scm

# Constants
scales = ["PAQ1", "PAQ2", "PAQ3", "PAQ4", "PAQ5", "PAQ6", "PAQ7", "PAQ8"]
eq_angles = [0, 45, 90, 135, 180, 225, 270, 315]
SCALES = scm.scale_abbrevs

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
FIGURE_DIR = PROJECT_DIR / "figures"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project directory: {PROJECT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Figure directory: {FIGURE_DIR}")

Project directory: /workspaces/SATP-Methodology
Output directory: /workspaces/SATP-Methodology/outputs
Figure directory: /workspaces/SATP-Methodology/figures


## Load Data

Load the SATP v1.5 dataset. Cantonese (yue) is already included in v1.5.
Exclude the LNC institution (Portuguese data from a different source).
Rename "Participant" to "participant" for soundscapy compatibility.

In [ ]:
satp = pd.read_excel(
    DATA_DIR / "SATP Dataset v1.5.xlsx",
    na_values=["", "N/A"],
)
satp = satp.rename(columns={"Participant": "participant"})
satp = satp[satp["Institution"] != "LNC"].copy()

languages = sorted(satp["Language"].unique())
print(f"Total rows (after LNC filter): {len(satp):,}")
print(f"Languages ({len(languages)}): {languages}")

## Step 1: Tracey's Circular Order Model

Tests whether the correlation matrices for each language conform to a circular order.
Criterion: CI > 0.70, p < 0.05.

In [ ]:
# Build list of DataFrames for rthor: overall SATP data first, then per language
matrices = [satp[scales].dropna()]
labels = ["SATP"]
for lang in languages:
    lang_data = satp[satp["Language"] == lang][scales].dropna()
    matrices.append(lang_data)
    labels.append(lang)

rthor_results = rthor.test(matrices, order="circular8", labels=labels)
rthor_results[["label", "ci", "p_value", "predictions", "agreements", "ties"]].round(3)

In [ ]:
# Determine pass/fail for Step 1
step1_langs = rthor_results.query("label != 'SATP'").copy()
step1_langs["pass"] = (step1_langs["ci"] > 0.70) & (step1_langs["p_value"] < 0.05)

pass_step1 = step1_langs.query("ci > 0.7")["label"].tolist()
fail_step1 = [l for l in languages if l not in pass_step1]

print(f"Pass Step 1 ({len(pass_step1)}): {pass_step1}")
print(f"Fail Step 1 ({len(fail_step1)}): {fail_step1}")

# Save Step 1 results
step1_out = step1_langs[["label", "ci", "p_value", "pass"]].rename(
    columns={"label": "language"}
)
step1_out.to_csv(OUTPUT_DIR / "step1_circular_order.csv", index=False)
print(f"Saved: {OUTPUT_DIR / 'step1_circular_order.csv'}")

# Filter dataset to passing languages for Step 2
satp = satp[satp["Language"].isin(pass_step1)].copy()

## Step 2: Structural Equation Modelling (CircE)

Fits four CircE model variants for each language using `soundscapy.satp.fit_circe`.
The function applies grand-mean ipsatization by default.
Thresholds: CFI >= 0.92, GFI >= 0.90, SRMR < 0.08.

In [ ]:
results_frames: list[CircEResults] = []
fit_errors: dict[str, Exception] = {}

for lang in sorted(satp["Language"].unique()):
    lang_data = satp[satp["Language"] == lang].copy()
    print(f"Fitting CircE for {lang}...", end=" ")
    try:
        results_frames.append(
            fit_circe(lang_data, language=lang, datasource="SATP", errors="warn")
        )
        print("OK")
    except Exception as e:
        fit_errors[lang] = e
        print(f"ERROR: {e}")

print(f"\nCompleted: {len(results_frames)}/{len(satp['Language'].unique())} languages")
if fit_errors:
    for lang, err in fit_errors.items():
        print(f"Error in {lang}: {err}")
else:
    print("No errors.")

In [ ]:
# Compile full SEM results table
full_table = pd.concat([r.table for r in results_frames], ignore_index=True)

# Save the full results (including angles) to CSV
full_table.to_csv(OUTPUT_DIR / "sem-fit-ipsatized.csv", index=False)
print(f"Saved: {OUTPUT_DIR / 'sem-fit-ipsatized.csv'}")

# Display condensed version
display_cols = [
    "language",
    "model",
    "n",
    "m",
    "chisq",
    "d",
    "p",
    "cfi",
    "gfi",
    "agfi",
    "srmr",
    "mcsc",
    "rmsea",
    "gdiff",
]
full_table[display_cols].round(5)

In [ ]:
# Calculate SEM fit scores (equal_com model only)
thresholds = {
    "CFI": 0.92,
    "GFI": 0.90,
    "SRMR": 0.08,
}
incl_in_score = ["CFI", "GFI", "SRMR"]
pass_thresh = 3

sem_res = full_table.copy()
sem_res["CFI_pass"] = sem_res["cfi"] >= thresholds["CFI"]
sem_res["GFI_pass"] = sem_res["gfi"] >= thresholds["GFI"]
sem_res["SRMR_pass"] = sem_res["srmr"] < thresholds["SRMR"]

sem_res["Score"] = sem_res[[x + "_pass" for x in incl_in_score]].sum(axis=1).astype(int)
sem_res["passing"] = pd.cut(
    sem_res["Score"], bins=[0, pass_thresh, 7], labels=["Fail", "Pass"], right=False
)

# Extract condensed scores for equal_com model
sem_scores = (
    sem_res[["language", "model", "n", "m", "cfi", "gfi", "srmr", "Score", "passing"]]
    .loc[sem_res["model"] == "equal_com"]
    .sort_values("language", ascending=True)
)
sem_scores.to_csv(OUTPUT_DIR / "sem-scores.csv", index=False)
print(f"Saved: {OUTPUT_DIR / 'sem-scores.csv'}")
sem_scores

In [ ]:
# Determine passing languages from Step 2
passing_step2 = (
    sem_res.loc[sem_res["model"] == "equal_com"]
    .query("passing != 'Fail'")["language"]
    .values
)
fail_step2 = [l for l in pass_step1 if l not in passing_step2]
print(f"Pass Step 2 ({len(passing_step2)}): {list(passing_step2)}")
print(f"Fail Step 2 ({len(fail_step2)}): {fail_step2}")

# Extract corrected angles from equal_com model
ang_df = sem_res[sem_res["model"] == "equal_com"][["language"] + scales].set_index(
    "language"
)
ang_dict = ang_df.T.to_dict(orient="list")

# Save adjusted angles
ang_df.to_csv(OUTPUT_DIR / "adjusted_angles.csv")
print(f"Saved: {OUTPUT_DIR / 'adjusted_angles.csv'}")
print("\nAdjusted angles per language:")
ang_df.round(2)

In [ ]:
# Filter dataset to Step 2 passing languages for Steps 3&4
satp = satp.query("Language in @passing_step2").copy()

## Steps 3 & 4: SSM Analysis and Congruence Testing

**Step 3**: SSM analysis using `circumplex.ssm_analyze()` to locate circumplex scales within each language's space. Criteria: R^2 > 0.80, amplitude > 0.15.

**Step 4**: Congruence testing using cosine similarity (Tucker's Congruence Coefficient) and Procrustes rotational distance. Threshold: > 0.90.

In [ ]:
from scipy.optimize import curve_fit
from scipy.spatial import procrustes as scipy_procrustes


def congruence(data1, data2, metric="cosine"):
    """Calculate congruence between two sets of circumplex coordinates."""
    from sklearn.metrics.pairwise import cosine_similarity
    from scipy.spatial.distance import cdist

    if metric == "cosine":
        sim = cosine_similarity(data1, data2)
    elif metric == "euclidean":
        sim = 1 - cdist(data1, data2, metric=metric)
    else:
        raise ValueError("metric must be 'cosine' or 'euclidean'")
    cong_vals = np.diag(sim)
    return np.mean(cong_vals), cong_vals


def procrustes_distance(
    data1, data2, procrustes_type="orthogonal", translate=True, scale=True
):
    """Calculate Procrustes distance between two coordinate sets."""
    if procrustes_type == "orthogonal":
        from procrustes import orthogonal

        pro_res = orthogonal(data1, data2, translate=translate, scale=scale)
    elif procrustes_type == "rotational":
        from procrustes import rotational

        pro_res = rotational(data1, data2, translate=translate, scale=scale)
    elif procrustes_type == "generic":
        from procrustes import generic

        pro_res = generic(data1, data2, translate=translate, scale=scale)
    return pro_res.error


def prepare_congruence_matrices(ssm_results, target_angles=eq_angles):
    """Prepare target and empirical coordinate matrices for congruence analysis."""
    data2 = ssm_results[["x_est", "y_est"]].values
    data1 = np.ones_like(data2)
    data1[:, 0] = np.cos(np.deg2rad(target_angles))
    data1[:, 1] = np.sin(np.deg2rad(target_angles))
    return data1, data2

In [ ]:
# Calculate overall per-recording means (reference circumplex)
overall_means = (
    satp.groupby("Recording")[scales]
    .mean()
    .rename(columns={s: f"{s}_ref" for s in scales})
    .reset_index()
)
ref_cols = [f"{s}_ref" for s in scales]

lang_rec_means = satp.groupby(["Language", "Recording"])[scales].mean().reset_index()


def test_language_locations(
    test_lang,
    test_angles,
    target_angles=eq_angles,
    scales=scales,
):
    """Test the congruence of the language locations with the target angles."""
    test_lang_means = lang_rec_means[lang_rec_means["Language"] == test_lang][
        ["Recording"] + list(scales)
    ]
    merged = test_lang_means.merge(overall_means, on="Recording")

    result = ssm_analyze(
        merged,
        scales=list(scales),
        angles=test_angles,
        measures=ref_cols,
        measures_labels=list(scales),
        boots=2000,
        seed=42,
    )

    data1, data2 = prepare_congruence_matrices(
        result.results, target_angles=target_angles
    )
    test_model_congruence, test_scale_congruences = congruence(
        data1, data2, metric="cosine"
    )
    pro_sim = 1 - procrustes_distance(data1, data2, procrustes_type="rotational")
    return result, test_model_congruence, test_scale_congruences, pro_sim


locating_corr_angles = {}
locating_eq_angles = {}

for test_lang in sorted(lang_rec_means["Language"].unique()):
    print(f"SSM analysis for {test_lang}...", end=" ")
    locating_eq_angles[test_lang] = test_language_locations(
        test_lang, test_angles=eq_angles, target_angles=eq_angles
    )
    locating_corr_angles[test_lang] = test_language_locations(
        test_lang,
        test_angles=ang_dict[test_lang],
        target_angles=eq_angles,
    )
    print("OK")

print(f"\nCompleted SSM analysis for {len(locating_corr_angles)} languages.")

In [ ]:
# Step 3 results: R-squared per scale
fit_results = pd.DataFrame.from_dict(
    {
        key: locating_corr_angles[key][0]
        .results.set_index("Label")["fit_est"]
        .to_dict()
        for key in locating_corr_angles.keys()
    }
).T
print("Step 3 - SSM R-squared per scale (corrected angles):")
fit_results.round(3)

In [ ]:
# Step 4 results: Congruence and Procrustes
congruence_df = pd.DataFrame(
    {
        "Language": list(locating_eq_angles.keys()),
        "Eq Ang Cosine": [x[1] for x in locating_eq_angles.values()],
        "Corr Ang Cosine": [x[1] for x in locating_corr_angles.values()],
        "Eq Ang Procrustes": [x[3] for x in locating_eq_angles.values()],
        "Corr Ang Procrustes": [x[3] for x in locating_corr_angles.values()],
    }
)

# Add per-scale R-squared from corrected angles
for s in scales:
    congruence_df[f"R2_{s}"] = [
        locating_corr_angles[lang][0].results.set_index("Label").loc[s, "fit_est"]
        for lang in congruence_df["Language"]
    ]

congruence_df.to_csv(OUTPUT_DIR / "step34_congruence.csv", index=False)
print(f"Saved: {OUTPUT_DIR / 'step34_congruence.csv'}")
print("\nStep 4 - Congruence results:")
congruence_df[
    [
        "Language",
        "Eq Ang Cosine",
        "Corr Ang Cosine",
        "Eq Ang Procrustes",
        "Corr Ang Procrustes",
    ]
].round(3)

## Confidence Tier Classification

Three-tier classification per language:
- **High**: Pass Steps 1, 2, 3 (all R^2 > 0.80), and 4 (cosine > 0.90)
- **Medium**: Pass Steps 1 and 2 but not all of 3&4
- **Low**: Fail Step 1 or 2

In [ ]:
# Build confidence tier table for all languages in the dataset
tier_rows = []
for lang in languages:
    row = {"language": lang}

    # Step 1
    s1 = step1_langs[step1_langs["label"] == lang]
    row["step1_ci"] = s1["ci"].values[0] if len(s1) > 0 else np.nan
    row["step1_pass"] = lang in pass_step1

    # Step 2
    s2 = sem_scores[sem_scores["language"] == lang]
    if len(s2) > 0:
        row["step2_score"] = s2["Score"].values[0]
        row["step2_pass"] = s2["passing"].values[0] == "Pass"
    else:
        row["step2_score"] = np.nan
        row["step2_pass"] = False

    # Steps 3&4
    if lang in locating_corr_angles:
        r2_vals = locating_corr_angles[lang][0].results["fit_est"].values
        row["step3_min_r2"] = np.min(r2_vals)
        row["step3_pass"] = np.all(r2_vals > 0.80)
        row["step4_cosine"] = locating_corr_angles[lang][1]
        row["step4_procrustes"] = locating_corr_angles[lang][3]
        row["step4_pass"] = locating_corr_angles[lang][1] > 0.90
    else:
        row["step3_min_r2"] = np.nan
        row["step3_pass"] = False
        row["step4_cosine"] = np.nan
        row["step4_procrustes"] = np.nan
        row["step4_pass"] = False

    # Tier assignment
    if not row["step1_pass"] or not row["step2_pass"]:
        row["tier"] = "Low"
    elif row["step3_pass"] and row["step4_pass"]:
        row["tier"] = "High"
    else:
        row["tier"] = "Medium"

    tier_rows.append(row)

confidence_tiers = pd.DataFrame(tier_rows)
confidence_tiers.to_csv(OUTPUT_DIR / "confidence_tiers.csv", index=False)
print(f"Saved: {OUTPUT_DIR / 'confidence_tiers.csv'}")

print("\nConfidence Tiers:")
for tier in ["High", "Medium", "Low"]:
    langs = confidence_tiers[confidence_tiers["tier"] == tier]["language"].tolist()
    print(f"  {tier} ({len(langs)}): {langs}")

confidence_tiers

## Figures

In [ ]:
# Figure 1: Mandarin with equal 45-degree angles
if "cmn" in locating_eq_angles:
    locating_eq_angles["cmn"][0].plot_circle(
        angle_labels=list(scales), title="cmn - equal angles"
    )
    plt.savefig(FIGURE_DIR / "cmn_eq_angles.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {FIGURE_DIR / 'cmn_eq_angles.png'}")
else:
    print("cmn not available for equal angles plot")

In [ ]:
# Figure 2: Mandarin with corrected angles
if "cmn" in locating_corr_angles:
    locating_corr_angles["cmn"][0].plot_circle(
        angle_labels=list(scales), title="cmn - corrected angles"
    )
    plt.savefig(FIGURE_DIR / "cmn_corr_angles.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {FIGURE_DIR / 'cmn_corr_angles.png'}")
else:
    print("cmn not available for corrected angles plot")

In [ ]:
# Figure 3: W06 comparison - scatter plot of High-confidence languages


def adj_iso_pl(values, angles, scale=None):
    iso_pl = np.sum(
        [np.cos(np.deg2rad(angle)) * values[i] for i, angle in enumerate(angles)]
    )
    if scale:
        iso_pl = iso_pl / (
            scale / 2 * np.sum(np.abs([np.cos(np.deg2rad(angle)) for angle in angles]))
        )
    return iso_pl


def adj_iso_ev(values, angles, scale=None):
    iso_ev = np.sum(
        [np.sin(np.deg2rad(angle)) * values[i] for i, angle in enumerate(angles)]
    )
    if scale:
        iso_ev = iso_ev / (
            scale / 2 * np.sum(np.abs([np.sin(np.deg2rad(angle)) for angle in angles]))
        )
    return iso_ev


def adj_angle_iso_coords(data, angles, scale=100):
    isopl = data.apply(
        lambda x: adj_iso_pl(x[scales].values, angles, scale=scale), axis=1
    )
    isoev = data.apply(
        lambda x: adj_iso_ev(x[scales].values, angles, scale=scale), axis=1
    )
    return isopl, isoev


high_langs = confidence_tiers[confidence_tiers["tier"] == "High"]["language"].tolist()

if high_langs:
    # Reload full dataset for this plot (need unfiltered satp for High-confidence languages)
    satp_full = pd.read_excel(
        DATA_DIR / "SATP Dataset v1.5.xlsx",
        na_values=["", "N/A"],
    )
    satp_full = satp_full.rename(columns={"Participant": "participant"})
    satp_full = satp_full[satp_full["Institution"] != "LNC"].copy()
    satp_full = satp_full[satp_full["Language"].isin(high_langs)].copy()

    w06 = satp_full.query("Recording == 'W06'")

    w06_res = {}
    for lang in high_langs:
        lang_data = w06.query("Language == @lang")
        if len(lang_data) > 0:
            isopl, isoev = adj_angle_iso_coords(lang_data, angles=ang_dict[lang])
            w06_res[lang] = {
                "Language": lang,
                "ISOPleasant": np.mean(isopl),
                "ISOEventful": np.mean(isoev),
            }

    w06_df = pd.DataFrame.from_dict(w06_res, orient="index")
    sspy.plotting.scatter(
        w06_df,
        hue="Language",
        s=70,
        title="W06 - High Confidence Languages (Adjusted Angles)",
    )
    plt.savefig(FIGURE_DIR / "W06_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {FIGURE_DIR / 'W06_comparison.png'}")
else:
    print("No High-confidence languages available for W06 plot")

In [ ]:
# Summary of all outputs
print("=" * 60)
print("OUTPUT FILES SUMMARY")
print("=" * 60)
for f in sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")
print()
for f in sorted(FIGURE_DIR.glob("*.png")):
    print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")
print("=" * 60)